In [1]:
import pandas as pd
import os

# Get the exact location of this notebook file
notebook_dir = os.getcwd()
print(f"Current Notebook Directory: {notebook_dir}")

# Go up steps until we reach the main project root folder
if ".ipynb_checkpoints" in notebook_dir:
    # Go up two levels if we are deep inside checkpoints
    project_root = os.path.abspath(os.path.join(notebook_dir, "..", ".."))
elif "notebooks" in notebook_dir:
    # Go up one level if we are in the standard notebooks folder
    project_root = os.path.abspath(os.path.join(notebook_dir, ".."))
else:
    project_root = notebook_dir

data_folder = os.path.join(project_root, "data")
print(f"Looking for data inside: {data_folder}")

# Check what files are inside the data folder
try:
    files = os.listdir(data_folder)
    print("✅ SUCCESS! Files inside data folder:", files)
    
    # If a CSV exists, let's try to preview the first 3 rows automatically
    csv_files = [f for f in files if f.endswith('.csv')]
    if csv_files:
        target_csv = os.path.join(data_folder, csv_files[0])
        print(f"\nLoading dataset: {csv_files[0]}...")
        df = pd.read_csv(target_csv)
        print(f"Dataset Loaded! Rows: {df.shape[0]} | Columns: {df.shape[1]}")
        display(df.head(3))
    else:
        print("\n⚠️ The data folder is empty or contains no CSV files. Make sure your dataset is placed inside it!")
except Exception as e:
    print(f"❌ Could not read the data folder. Error: {e}")

Current Notebook Directory: d:\spotify_recommendation_system\notebooks
Looking for data inside: d:\spotify_recommendation_system\data
✅ SUCCESS! Files inside data folder: ['raw_spotify_data.csv']

Loading dataset: raw_spotify_data.csv...
Dataset Loaded! Rows: 149860 | Columns: 11


,spotify_track_uri,ts,platform,ms_played,track_name,artist_name,album_name,reason_start,reason_end,shuffle,skipped
0,2J3n32GeLmMjwuAzyhcSNe,2013-07-08 02:44:34,web player,3185,"Say It, Just Say It",The Mowgli's,Waiting For The Dawn,autoplay,clickrow,False,False
1,1oHxIPqJyvAYHy0PVrDU98,2013-07-08 02:45:37,web player,61865,Drinking from the Bottle (feat. Tinie Tempah),Calvin Harris,18 Months,clickrow,clickrow,False,False
2,487OPlneJNni3NWC8SYqhW,2013-07-08 02:50:24,web player,285386,Born To Die,Lana Del Rey,Born To Die - The Paradise Edition,clickrow,unknown,False,False


In [2]:
print("--- Step 5: High-Level Dataset Profiling ---")

# 1. Structural Summary (Check data types)
print("\n[INFO] Data Structure Profile:")
df.info()

# 2. Check for Missing Data points
print("\n[INFO] Missing Value Counts:")
print(df.isnull().sum())

# 3. Check how many unique tracks and artists you have
print("\n[INFO] Unique Track & Artist Footprint:")
print(f"Unique Tracks: {df['spotify_track_uri'].nunique()}")
print(f"Unique Artists: {df['artist_name'].nunique()}")

--- Step 5: High-Level Dataset Profiling ---

[INFO] Data Structure Profile:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 149860 entries, 0 to 149859
Data columns (total 11 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   spotify_track_uri  149860 non-null  object
 1   ts                 149860 non-null  object
 2   platform           149860 non-null  object
 3   ms_played          149860 non-null  int64 
 4   track_name         149860 non-null  object
 5   artist_name        149860 non-null  object
 6   album_name         149860 non-null  object
 7   reason_start       149717 non-null  object
 8   reason_end         149743 non-null  object
 9   shuffle            149860 non-null  bool  
 10  skipped            149860 non-null  bool  
dtypes: bool(2), int64(1), object(8)
memory usage: 10.6+ MB

[INFO] Missing Value Counts:
spotify_track_uri      0
ts                     0
platform               0
ms_played              

In [3]:
print("--- Step 6: Transforming Features ---")

# 1. Convert timestamp text to a proper datetime format
df['ts'] = pd.to_datetime(df['ts'])

# 2. Convert milliseconds played to decimal minutes for easier threshold filtering
df['minutes_played'] = df['ms_played'] / 60000

# 3. Create an explicit "Listen Completion Rate" feature
# If a user skipped it immediately vs listening to a solid portion
print("Data transformation complete!")
print("\nNew column types:")
print(df[['ts', 'minutes_played']].dtypes)

# Preview the top 5 most frequently played artists in the history
print("\n--- Your Top 5 Most Played Artists ---")
print(df['artist_name'].value_counts().head(5))

--- Step 6: Transforming Features ---
Data transformation complete!

New column types:
ts                datetime64[ns]
minutes_played           float64
dtype: object

--- Your Top 5 Most Played Artists ---
artist_name
The Beatles       13621
The Killers        6878
John Mayer         4855
Bob Dylan          3814
Paul McCartney     2697
Name: count, dtype: int64


In [4]:
print("--- Step 7: Deep User Behavior Analysis ---")

# 1. Calculate general skip statistics using the dataset's 'skipped' column
skip_counts = df['skipped'].value_counts()
print("[INFO] Explicit Skip Distribution:")
print(skip_counts)

# 2. Extract the hour of the day from our newly fixed datetime column
df['hour'] = df['ts'].dt.hour

# Find the top 3 peak listening hours of the day
print("\n--- Top 3 Peak Listening Hours ---")
peak_hours = df['hour'].value_counts().head(3)
for hour, count in peak_hours.items():
    # Format to readable AM/PM style
    ampm_hour = f"{hour:02d}:00"
    print(f"Time: {ampm_hour} | Total Songs Streamed: {count}")

# 3. Filter down to see how often tracks are played fully vs abandoned early
short_plays = df[df['minutes_played'] < 0.5].shape[0]
print(f"\n[INFO] Short Streams (< 30 seconds): {short_plays} tracks ({short_plays/len(df)*100:.1f}%)")

--- Step 7: Deep User Behavior Analysis ---
[INFO] Explicit Skip Distribution:
skipped
False    141991
True       7869
Name: count, dtype: int64

--- Top 3 Peak Listening Hours ---
Time: 00:00 | Total Songs Streamed: 10884
Time: 23:00 | Total Songs Streamed: 10516
Time: 20:00 | Total Songs Streamed: 10494

[INFO] Short Streams (< 30 seconds): 55660 tracks (37.1%)


In [5]:
print("--- Step 8: Computing Recommendation Target Scores ---")

# 1. Label each stream entry: 1 for a good listen, -1 for an implicit/explicit skip
df['listen_weight'] = df.apply(
    lambda row: -1.5 if (row['minutes_played'] < 0.5 or row['skipped'] == True) else 1.0, 
    axis=1
)

# 2. Aggregate by artist to find total score and frequency
artist_profiles = df.groupby('artist_name').agg(
    total_streams=('listen_weight', 'count'),
    preference_score=('listen_weight', 'sum')
).reset_index()

# 3. Sort by our new preference score to find the true optimized favorites
top_rated_artists = artist_profiles.sort_values(by='preference_score', ascending=False)

print("\n✅ Preference matrix calculated successfully!")
print("\n--- Optimized Top 5 Artists (Adjusted for Skip Penalties) ---")
display(top_rated_artists.head(5))

print("\n--- Least Favorite Artists (Most Heavily Skipped/Abandoned) ---")
display(artist_profiles.sort_values(by='preference_score', ascending=True).head(5))

--- Step 8: Computing Recommendation Target Scores ---

✅ Preference matrix calculated successfully!

--- Optimized Top 5 Artists (Adjusted for Skip Penalties) ---


,artist_name,total_streams,preference_score
3602,The Killers,6878,1380.5
1511,Howard Shore,1446,746.0
3708,The Strokes,1818,618.0
1115,Ennio Morricone,994,601.5
1728,Joaquín Sabina,796,543.5



--- Least Favorite Artists (Most Heavily Skipped/Abandoned) ---


,artist_name,total_streams,preference_score
2093,Led Zeppelin,2482,-1070.5
3500,The Beatles,13621,-1041.5
3003,Radiohead,2305,-1022.5
3503,The Black Keys,2231,-801.5
3727,The Velvet Underground,1507,-515.5


In [7]:
print("--- Step 9: Training the Recommendation Model ---")

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error

# 1. Encode artist names into numeric IDs for the model
artist_encoder = LabelEncoder()
artist_profiles['artist_id'] = artist_encoder.fit_transform(artist_profiles['artist_name'])

# 2. Prepare features (X) and our calculated target score (y)
X = artist_profiles[['artist_id']].values
y = artist_profiles['preference_score'].values

# 3. Split data: 80% to train the model, 20% to test how accurate it is
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"[INFO] Training Samples: {len(X_train)} | Testing Samples: {len(X_test)}")

# 4. Initialize a Baseline Regression Model to learn the patterns
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=42)

print("Training the model now... Please wait...")
model.fit(X_train, y_train)
print("✅ Model training complete!")

# 5. Evaluate the model's accuracy on unseen testing data
predictions = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
print(f"\n[METRIC] Root Mean Squared Error (RMSE): {rmse:.4f}")
print("(A lower RMSE means our model's predictions are highly accurate!)")

--- Step 9: Training the Recommendation Model ---
[INFO] Training Samples: 3290 | Testing Samples: 823
Training the model now... Please wait...
✅ Model training complete!

[METRIC] Root Mean Squared Error (RMSE): 57.7156
(A lower RMSE means our model's predictions are highly accurate!)


In [8]:
print("--- Step 10: Generating Best Machine Learning Recommendations ---")

# 1. Use the trained model to predict scores for all artists
artist_profiles['predicted_preference'] = model.predict(X)

# 2. Sort by the model's predicted score
ml_recommendations = artist_profiles.sort_values(by='predicted_preference', ascending=False)

# 3. Filter out artists that have heavily negative historical scores to ensure accuracy
top_ml_picks = ml_recommendations[ml_recommendations['preference_score'] > 0].head(10)

print("\n🎯 THE MACHINE LEARNING RECOMMENDATION RESULTS:")
print("Here are the top 10 recommended artists optimized by your Random Forest Model:")
print("=" * 70)

for rank, (idx, row) in enumerate(top_ml_picks.iterrows(), 1):
    print(f"Rank {rank:02d} | Artist: {row['artist_name']:<25} | Model Confidence Score: {row['predicted_preference']:.2f}")
print("=" * 70)

--- Step 10: Generating Best Machine Learning Recommendations ---

🎯 THE MACHINE LEARNING RECOMMENDATION RESULTS:
Here are the top 10 recommended artists optimized by your Random Forest Model:
Rank 01 | Artist: The Killers               | Model Confidence Score: 882.89
Rank 02 | Artist: The Strokes               | Model Confidence Score: 396.01
Rank 03 | Artist: Joe Bataan                | Model Confidence Score: 337.95
Rank 04 | Artist: Joaquín Sabina            | Model Confidence Score: 337.54
Rank 05 | Artist: John Mayer                | Model Confidence Score: 328.57
Rank 06 | Artist: Enrique Granados          | Model Confidence Score: 325.21
Rank 07 | Artist: Ennio Morricone           | Model Confidence Score: 322.65
Rank 08 | Artist: Enjambre                  | Model Confidence Score: 317.98
Rank 09 | Artist: Joaquin Sabina y Viceversa | Model Confidence Score: 168.71
Rank 10 | Artist: Keuning                   | Model Confidence Score: 167.95


In [9]:
print("--- Step 11: Training Industry-Grade SVD Model ---")

from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split as surprise_split
from surprise import accuracy
import pandas as pd

# 1. Format data for the specialized ML engine (User, Item, Rating)
df_ml = artist_profiles[['artist_name', 'preference_score']].copy()
df_ml['user_id'] = 'ME'

# Normalize scores between 1 and 10 to stabilize SVD learning gradients
min_score = df_ml['preference_score'].min()
max_score = df_ml['preference_score'].max()
df_ml['scaled_score'] = 1 + 9 * (df_ml['preference_score'] - min_score) / (max_score - min_score)

# 2. Load dataset into Surprise framework
reader = Reader(rating_scale=(1, 10))
data = Dataset.load_from_df(df_ml[['user_id', 'artist_name', 'scaled_score']], reader)

# 3. Split 80/20 using the correct module
trainset, testset = surprise_split(data, test_size=0.2, random_state=42)

# 4. Initialize and Train Latent Factor SVD
svd_model = SVD(n_factors=20, random_state=42)  # 20 hidden musical traits
print("Training Latent Factor SVD model...")
svd_model.fit(trainset)
print("✅ SVD Training complete!")

# 5. Check accuracy
predictions = svd_model.test(testset)
rmse = accuracy.rmse(predictions)

--- Step 11: Training Industry-Grade SVD Model ---
Training Latent Factor SVD model...
✅ SVD Training complete!
RMSE: 0.3002


In [10]:
print("--- Step 12: Generating Advanced SVD Recommendations ---")

# 1. Predict the scaled score for every unique artist in your database
all_artists = df_ml['artist_name'].unique()
svd_predictions = []

for artist in all_artists:
    # Estimate the rating for user 'ME' on a scale of 1 to 10
    pred = svd_model.predict(uid='ME', iid=artist)
    svd_predictions.append({
        'artist_name': artist,
        'predicted_scaled_rating': pred.est
    })

# 2. Convert to DataFrame and merge with actual data for validation
svd_pred_df = pd.DataFrame(svd_predictions)
svd_final = pd.merge(svd_pred_df, artist_profiles, on='artist_name')

# 3. Filter out highly penalized artists to guarantee quality recommendations
top_svd_picks = svd_final[svd_final['preference_score'] > 0].sort_values(by='predicted_scaled_rating', ascending=False).head(10)

print("\n🎯 THE INDUSTRY-GRADE SVD RECOMMENDATION RESULTS:")
print("Top 10 recommended artists optimized by your Latent Factor Model:")
print("=" * 75)
for rank, (idx, row) in enumerate(top_svd_picks.iterrows(), 1):
    print(f"Rank {rank:02d} | Artist: {row['artist_name']:<30} | SVD Predicted Rating: {row['predicted_scaled_rating']:.2f}/10.0")
print("=" * 75)

--- Step 12: Generating Advanced SVD Recommendations ---

🎯 THE INDUSTRY-GRADE SVD RECOMMENDATION RESULTS:
Top 10 recommended artists optimized by your Latent Factor Model:
Rank 01 | Artist: Howard Shore                   | SVD Predicted Rating: 5.20/10.0
Rank 02 | Artist: The Strokes                    | SVD Predicted Rating: 5.16/10.0
Rank 03 | Artist: Joaquín Sabina                 | SVD Predicted Rating: 5.12/10.0
Rank 04 | Artist: Jorge Drexler                  | SVD Predicted Rating: 5.07/10.0
Rank 05 | Artist: Paul McCartney                 | SVD Predicted Rating: 5.04/10.0
Rank 06 | Artist: Kings of Leon                  | SVD Predicted Rating: 5.03/10.0
Rank 07 | Artist: Cage The Elephant              | SVD Predicted Rating: 5.02/10.0
Rank 08 | Artist: Giacomo Puccini                | SVD Predicted Rating: 5.02/10.0
Rank 09 | Artist: Ed Maverick                    | SVD Predicted Rating: 5.01/10.0
Rank 10 | Artist: Houndmouth                     | SVD Predicted Rating: 5.01/10

In [11]:
print("--- Step 13: Final Interactive Recommendation Function ---")

import numpy as np

def get_similar_artists(target_artist, top_n=5):
    """
    Finds and returns the top_n most similar artists based on learned SVD latent traits.
    """
    # 1. Verify if the artist exists in our dataset
    if target_artist not in df_ml['artist_name'].values:
        return f"⚠️ Artist '{target_artist}' not found in listening history records."
    
    # 2. Extract the trained internal representation (vector) for the target artist
    try:
        target_inner_id = trainset.to_inner_iid(target_artist)
        target_vector = svd_model.qi[target_inner_id]
    except ValueError:
        return f"⚠️ Artist '{target_artist}' wasn't included in the model training partition stage."
    
    # 3. Calculate similarity distances against every other artist vector in memory
    similarities = []
    for artist_name in trainset.all_items():
        raw_name = trainset.to_raw_iid(artist_name)
        if raw_name == target_artist:
            continue
            
        current_vector = svd_model.qi[artist_name]
        
        # Compute Cosine Similarity between the hidden trait vectors
        dot_product = np.dot(target_vector, current_vector)
        norm_target = np.linalg.norm(target_vector)
        norm_current = np.linalg.norm(current_vector)
        
        cosine_sim = dot_product / (norm_target * norm_current) if (norm_target * norm_current) > 0 else 0
        similarities.append((raw_name, cosine_sim))
        
    # 4. FIXED: Using reverse=True for standard Python list sorting
    similarities.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\n🎯 Because you listen to '{target_artist}', your SVD Model recommends:")
    print("-" * 65)
    for rank, (name, score) in enumerate(similarities[:top_n], 1):
        # Scale score to percentage for a clean UI presentation
        match_percentage = (score + 1) / 2 * 100
        print(f"Match {rank:02d}: {name:<35} | Vector Match: {match_percentage:.1f}%")
    print("-" * 65)

# --- TEST THE SYSTEM ---
get_similar_artists("The Strokes", top_n=5)

--- Step 13: Final Interactive Recommendation Function ---

🎯 Because you listen to 'The Strokes', your SVD Model recommends:
-----------------------------------------------------------------
Match 01: The Clash                           | Vector Match: 87.9%
Match 02: Karen Dalton                        | Vector Match: 83.7%
Match 03: Jumbo                               | Vector Match: 81.5%
Match 04: Los Tucanes De Tijuana              | Vector Match: 80.9%
Match 05: Graciela Flores                     | Vector Match: 78.3%
-----------------------------------------------------------------
